(ch:unsupervisedLearning)=
# 비지도 학습

:::{note} 감사의 글

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 이미지 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.
:::

:::{seealso} 코드 실행
[(코드 워크아웃) 비지도 학습](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-unsupervised_learning.ipynb)을 병행하여 읽을 것을 권장한다.
:::

비지도 학습은 레이블이 없는 데이터를 학습하는 기법이다.
[PCA(주성분 분석)](#ch:pca)에서 다룬 차원 축소 기법도 비지도 학습의 전형적인 예제이다.
여기서는 다음 주제와 관련된 비지도 학습을 다룬다.

- 군집화: 비슷한 샘플끼리의 군집 형성. 고객 분류, 추천 시스템, 검색 엔진, 이미지 분할 등.

- 이상치 탐지: 정상 데이터와 이상치 구분. 생산라인에서 결함 제품 탐지 등.

- 데이터 밀도 추정: 데이터셋의 확률 밀도 추정. 이상치 분류, 데이터 시각화 등.

## 분류 대 군집화

**군집**<font size='2'>cluster</font>은 주어진 특성에 비추어 서로 유사한 대상들의 모음을 가리킨다.
예를 들어 이름을 모르는 꽃들의 꽃잎 길이와 너비를 측정했을 때,
측정값이 서로 비슷한 꽃들이 하나의 군집을 이룰 수 있다.
**군집화**<font size='2'>clustering</font>는 레이블이 없는 데이터를
특성의 유사성이나 데이터 분포의 구조에 따라 여러 군집으로 나누는 과정이다.

분류와 군집화는 각 샘플에 하나의 그룹을 부여할 수 있다는 점에서 유사하다.
하지만 분류는 미리 주어진 클래스 레이블을 예측하는 지도 학습 문제인 반면,
군집화는 미리 주어진 레이블 없이 데이터에 내재된 유사성이나 밀도 구조를 이용하여
샘플들을 여러 군집으로 구분하는 비지도 학습 문제이다.

다음 세 가지 방법을 자세히 소개한다.
방법에 따라 포착하기 쉬운 군집의 형태와 가정이 다르다.

* k-평균: 각 군집의 평균 위치인 센트로이드를 중심으로 가까운 샘플들을 묶는 방법
* DBSCAN: 밀도가 높은 영역에서 서로 밀도 도달 가능한 샘플들을 묶고, 나머지 일부 샘플을 잡음으로 식별하는 방법
* 가우스 혼합 모델: 데이터가 여러 가우스 분포의 혼합에서 생성되었다고 가정하고, 각 성분에 속할 가능성을 이용하여 군집화하는 확률 모델

**예제: 붓꽃 데이터셋 군집화**

아래 왼쪽 그림은 붓꽃의 꽃잎 길이와 너비를 특성으로 사용하여
각 샘플의 실제 품종 레이블을 표시한 것이다.
세토사(setosa), 버시컬러(versicolor), 버지니카(virginica)의 세 품종이 표시되어 있다.

반면 오른쪽 그림은 품종 레이블을 사용하지 않고,
샘플들이 이루는 군집의 형태만 꽃잎 길이와 너비 기준으로 구분하여 표시한 것이다.
세토사에 해당하는 샘플들은 꽃잎의 길이와 너비가 다른 샘플들과 뚜렷하게 달라
독립된 군집으로 잘 구분된다.
하지만 버시컬러와 버지니카에 해당하는 샘플들은 서로 엉켜 있어
두 개의 군집으로 뚜렷하게 분리되지 않는다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-01.png" width="80%"/></div>

반면에 **가우스 혼합 모델**<font size='2'>Gaussian Mixture Model</font>(GMM)을 
꽃잎의 길이와 너비 뿐만 아니라
꽃받침의 길이와 너비 특성까지 특성으로 사용하는 붓꽃 데이터셋에 대해 적용하면
세 개의 군집을 약 97% 정도 정확하게 생성한다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-02.png" width="450"/></div>

## k-평균

군집의 중심인 센트로이드<font size='2'>centroid</font>를 몇 개 찾은 다음 
각 센트로이드에 가깝게 위치한 샘플들로 구성된 군집을 형성하는 기법이다.

**사이킷런의 `KMeans` 모델** 

아래 그림은 다섯 개의 샘플 덩어리로 이루어진 데이터셋을 보여준다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-03.png" width="450"/></div>

위 데이터셋에 대해 다섯 개의 군집을 형성하는 k-평균 알고리즘은 다음과 같이 적용한다.
군집 수를 몇 개로 지정하는 게 가장 좋은지는 미리 알 수 없다. 
나중에 몇 개의 군집이 적절한가를 판단하는 여러 방식을 살펴볼 것이다. 

아래 코드에서 `X`는 위 산점도에 포함된 데이터 샘플들로 구성된 훈련셋을 가리킨다.

```python
from sklearn.cluster import KMeans

k = 5
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
y_pred = kmeans.fit_predict(X)
```

**예측값**

`predict()` 함수의 반환값은 0, 1, 2, 3, 4 등 정수로 구성된다.
하지만 이는 임의로 지정된 군집의 인덱스를 가리킬 뿐이며 클래스 분류와는 아무 상관 없다.

```python
>>> y_pred
array([4, 0, 1, ..., 2, 1, 0])
```

**센트로이드 정보**

`KMeans` 모델이 찾아낸 센트로이드 정보는 `cluster_centers_` 속성에 저장된다.

```python
>>> kmeans.cluster_centers_
array([[-2.80389616,  1.80117999],
       [ 0.20876306,  2.25551336],
       [-2.79290307,  2.79641063],
       [-1.46679593,  2.28585348],
       [-2.80037642,  1.30082566]])
```

**보로노이 다이어그램**

**보로노이 다이어그램**<font size='2'>Voronoi diagram</font>은
평면을 특정 점(센트로이드)까지의 거리가 가장 가까운 점들의 집합으로 분할한 그림이다. 
점들이 군집을 잘 구성하는지 여부를 쉽게 확인할 수 있다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-04.png" width="450"/></div>

왼쪽 상단 군집에 포함된 샘플들 중에서  군집 경계 근처에 있는 샘플들의 군집이 잘못 지정됐다.
이유는 그 오른편에 위치한 군집의 직경이 보다 크기 때문에 사실
그 군집에 속해야 하는 샘플이 왼쪽 센트로이드와의 거리가 단지 보다 가깝다는 이유로 왼쪽 군집으로 판정되었다.
이렇듯 군집의 직경이 서로 많이 다르면 군집화가 잘 작동하지 않을 수 있다.

**하드 군집화 대 소프트 군집화**

지금까지 살펴보았듯이 k-평균 모델 객체의 `labels_` 속성은 훈련 샘플마다
가장 가까운 센트로이드가 나타내는 군집의 인덱스를 저장한다.
또한 `predict()` 메서드는 새로운 샘플에 대해 가장 가까운 센트로이드의 군집 인덱스를 반환한다.
여기서 군집 인덱스는 군집을 구별하기 위해 임의로 부여된 번호일 뿐,
클래스 레이블과 같은 의미를 갖지 않는다.
이처럼 샘플마다 하나의 군집만 할당하는 방식을 **하드 군집화**<font size='2'>hard clustering</font>라 한다.

한편 샘플과 여러 군집 사이의 관계를 연속적인 점수로 나타낼 수도 있다.
예를 들어 각 센트로이드까지의 거리 또는 가우스 방사 기저 함수(RBF)를 이용한
유사도 값을 계산하여 새로운 특성으로 사용할 수 있다.
이러한 특성은 샘플이 여러 센트로이드와 얼마나 가깝거나 유사한지를 함께 나타낸다.

사이킷런의 `KMeans` 모델에서 `transform()` 메서드는 각 샘플과 모든 센트로이드 사이의
유클리드 거리를 계산하여 반환한다.
반환되는 배열의 열 하나는 센트로이드 하나에 대응하며,
값이 작을수록 해당 센트로이드에 가까움을 의미한다.

```python
>>> kmeans.transform(X_new).round(2)
array([[2.81, 0.33, 2.9 , 1.49, 2.89],
       [5.81, 2.8 , 5.85, 4.48, 5.84],
       [1.21, 3.29, 0.29, 1.69, 1.71],
       [0.73, 3.22, 0.36, 1.55, 1.22]])
```

반면에 [머신러닝 프로젝트](#ch:end2end)에서 `KMeans` 모델을 상속하는 형식으로 정의된 
`ClusterSimilarity` 클래스의 `transform()` 메서드는 가우스 방사 기저 함수인 `rbf_kernel()` 함수를 이용하여 각 샘플에 대해 모든 센트로이드들과의 유사도 점수를 계산한다.
계산된 점수는 캘리포니아 주택 가격 예측 모델의 훈련에 사용되도록 새로운 특성으로 추가되었다.

```python
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None): # sample_weight: 샘플별로 가중치 적용
        self.kmeans_ = KMeans(self.n_clusters, n_init=10, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # 항상 self 반환

    # 구역 데이터 샘플과 각 센트로이드 사이의 유사도 측정
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    ```

:::{admonition} 가우스 방사 기저 함수
:class: info

`rbf_kernel()` 는 **가우스 방사 기저 함수**<font size='2'>Gaussian radial basis function</font>를 
가리키며 다음과 같이 정의된다.
특정 지점을 가리키는 랜드마크<font size='2'>landmark</font>인 $\mathbf{m}$으로부터 조금만 멀어져도 
함숫값이 급격히 작아진다. 

$$
\phi(\mathbf{x},\mathbf{m}) = \exp \left( -\gamma \|\mathbf{x} - \mathbf{m} \|^2 \right)
$$

하이퍼파라미터인 **감마**($\gamma$, gamma)는 데이터 샘플이 랜드마크로부터 멀어질 때
가우스 RBF 함수의 반환값이 얼마나 빠르게 0에 수렴하도록 하는가를 결정한다.
감마 값이 클수록 랜드마크로부터 조금만 멀어져도 보다 빠르게 0에 수렴한다.
따라서 가우스 RBF 함수의 그래프가 보다 좁은 종 모양을 띤다.

아래 그래프는 감마가 1일 때와 0.01 때의 차이를 명확하게 보여준다.
즉 랜드마크인 $\mathbf{m}=$ 0으로부터 거리가 멀어질 때 감마가 1이면 매우 급격하게 함숫값이 0으로 줄어든다.
즉, 랜드마크로부터 조금만 멀어져도 유사도가 매우 약해진다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/rbf_kernel.png" width="400"></div>
<br>

예를 들어, 유사도를 도심 상권의 발달 정도로 해석할 때 도시 중심으로 조금만 멀어져도
상권이 좋지 않음을 의미한다. 대표적으로 소도시의 상권을 잘 반영한다.
반면에 서울의 경우 도시 중심으로부터 조금 떨어져 있다 하더라도 상권이 충분히 잘 발달되어 있을 수 있다.
따라서 그런 경우에는 감마를 0.01처럼 작게 지정해야 한다.

`ClusterSimilarity` 클래스의 `transform()` 메서드는 각 구역과 각 센트로이드 사이의
가우스 RBF 값을 계산하여, 센트로이드에 대한 유사도를 새로운 특성으로 만든다.

위 코드에서는 `fit()`을 실행할 때 구역별 중앙 주택 가격을 `sample_weight`로 전달한다.
따라서 주택 가격이 높은 구역일수록 k-평균 목적 함수와 센트로이드 위치의 결정에 더 큰 영향을 준다.
그 결과 센트로이드는 고가 주택 구역들이 많이 반영된 위치 쪽으로 이동하는 경향이 있다.
단, 센트로이드는 실제 구역 샘플이 아니라 여러 샘플의 가중 평균 위치일 수 있다.

이렇게 얻은 센트로이드와의 RBF 유사도는 주택 가격 예측에 활용할 새로운 위치 기반 특성이 된다.
즉, 여기서의 군집화는 레이블 없이 구조만 발견하려는 순수한 비지도 학습이라기보다,
주택 가격이라는 타깃 정보를 가중치로 사용한 특성 공학 과정이다.

위 코드에서는 `gamma=1`로 지정하였으므로 센트로이드에 가까운 구역에만 높은 유사도가 부여된다.
`gamma`를 작게 지정하면 센트로이드에서 더 멀리 떨어진 구역에도 비교적 높은 유사도가 부여된다.
:::

### k-평균 알고리즘

먼저 생성할 군집의 개수를 나타내는 $k$ 값을 지정한다.
초기 센트로이드 $k$개가 정해지면, 센트로이드의 변화가 충분히 작아지거나
지정된 최대 반복 횟수에 도달할 때까지 다음 두 단계를 반복한다.

* 각 샘플을 가장 가까운 센트로이드에 할당하여 $k$개의 군집을 구성한다.
* 각 군집에 할당된 샘플들의 평균을 계산하여 해당 군집의 새로운 센트로이드로 지정한다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-05.png" width="700"/></div>

**무작위 초기화 문제**

임의로 선택된 초기 센트로이드에 따라 매우 다른 모양과 성질의 군집이 생성될 수 있다.
아래 오른쪽 그림은 센트로이드 초기화가 다르면 최종 결과가 많이 다를 수 있음을 잘 보여준다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-06.png" width="750"/></div>

**센트로이드 초기화 반복 횟수**

무작위 초기화 문제를 줄이기 위해 k-평균 알고리즘의 초기화를 여러 번 실행한 다음에 가장 낮은 
관성을 보이는 모델을 최종 모델로 선택할 수 있다.
초기화 반복 횟수는 사용되는 초기화 알고리즘에 따라 달라진다.
사이킷런의 `KMeans` 모델은 기본적으로 보다 안정적인 초기화를 위해 `k-means++` 방식을 사용한다.

:::{note} 관성

**관성**<font size='2'>inertia</font>은 각 샘플과 그 샘플이 속한 군집의 센트로이드 사이의
제곱 거리의 합이다. k-평균에서는 각 샘플을 가장 가까운 센트로이드가 나타내는 군집에
할당하므로, 관성은 각 샘플과 가장 가까운 센트로이드 사이의 제곱 거리의 합과 같다.
관성이 작을수록 각 군집의 샘플들이 센트로이드 주변에 조밀하게 모여 있음을 의미한다.

훈련된 `KMeans` 모델의 경우 `inertia_` 속성에 훈련셋에 대한 관성 값이 저장된다.
또한 훈련셋에 대해 `score()` 메서드를 호출하면 관성의 음숫값을 반환한다.
이는 사이킷런의 점수 체계에서 반환값이 클수록 더 좋은 모델을 나타내도록 하기 때문이다.
초기화를 여러 번 실행하도록 설정된 `KMeans` 모델은 각 실행 결과 중
관성이 가장 작은 군집화 결과를 최종 결과로 선택한다.
:::

### 최적의 군집수

군집수가 적절하지 않으면 좋지 않은 모델로 수렴할 수 있다.
5개의 군집이 적절한 데이터셋에 대해 왼쪽 그림처럼 3개의 센트로이드만 사용하거나,
오른쪽 그림처럼 너무 많은 8개의 센트로이드를 사용하면 군집화가 적절하게
진행되지 않는다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-08.png" width="700"/></div>

**방법 1: 관성과 군집수**

군집수 k가 증가할 수록 관성은 기본적으로 줄어들기에 관성만으로 모델을 평가하기엔 부족하다.
하지만 관성이 더 이상 획기적으로 줄어들지 않는 지점을 군집수 후보로 선정할 수는 있다.
예를 들어 아래 그래프는 k가 1부터 9까지 변하는 동안 훈련된 모델의 관성을 측정하며,
관성이 현격하게 줄어드는 현상이 약화되기 시작하는 k=4가 군집수 후보로 괜찮아 보인다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-09.png" width="600"/></div>

이유는 군집이 네 개보다 작으면 별로이고, 4개보다 많아도 훨씬 좋아진다고 보기 어렵기 때문이다.
하지만 4개의 군집으로 구성하려 하면 아래 그림과 같이 왼쪽 하단 두 개의 군집이 하나의 군집으로 처리될 수 있기에
가장 좋은 군집화라고 말하기 어렵다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-10.png" width="400"/></div>

**방법 2: 실루엣 점수와 군집수**

**실루엣 점수**<font size='2'>silhouette score</font>는 모든 샘플의 실루엣 계수의 평균값이다.
샘플의 **실루엣 계수**<font size='2'>silhouette coefficient</font>는 다음 식으로 계산된다.

$$
\frac{b - a}{\max(a, b)}
$$

- $a$: 해당 샘플과 동일한 군집에 속하는 다른 샘플들 사이의 거리의 평균값
- $b$: 해당 샘플이 속하지 않은 각 군집에 대해 계산한 평균 거리 중 최솟값.
  즉, 평균 거리가 가장 작은 다른 군집과 해당 샘플 사이의 평균 거리

실루엣 계수는 군집 수가 2개 이상이고 샘플 수보다 작을 때 정의되며,
-1과 1 사이의 값을 갖는다.

* 1에 가까운 값: 동일 군집의 샘플들과는 가깝고 다른 군집들과는 멀리 떨어져 있음을 의미한다.
* 0에 가까운 값: 해당 샘플이 두 군집의 경계 부근에 있거나 군집들이 서로 겹쳐 있음을 의미한다.
* -1에 가까운 값: 해당 샘플이 현재 군집보다 다른 군집에 더 가까워 잘못 할당되었을 가능성이 높음을 의미한다.

k=4가 여전히 매우 좋아 보인다. 
하지만 관성의 경우와는 달리 k=5도 역시 꽤 좋다는 것을 알 수 있다. 

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-11.png" width="600"/></div>

**방법 3: 실루엣 다이어그램과 군집수**

**실루엣 다이어그램**은 군집별로 샘플의 실루엣 계수를 모아 표시한 그래프다.
군집별로 실루엣 계수를 정렬하여 그리면 아래 그림처럼 여러 개의 칼날 모양이 형성된다.

- 칼날 두께: 해당 군집에 포함된 샘플 수
- 칼날의 가로 길이: 해당 샘플의 실루엣 계수
- 빨간 파선: 실루엣 점수

좋은 군집화 결과에서는 실루엣 점수가 크고, 대부분의 샘플이 충분히 큰 양의 실루엣 계수를 갖는다.
반대로 실루엣 계수가 0에 가까운 샘플이 많으면 군집 경계가 불분명하다는 뜻이며,
음수인 샘플이 많으면 일부 샘플이 적절하지 않은 군집에 할당되었을 가능성이 높다.

칼날의 두께는 군집별 크기를 비교하는 데 사용할 수 있다.
데이터의 실제 군집들이 비슷한 크기라고 기대되는 경우에는,
칼날 두께가 크게 불균형한 모델보다 비슷한 두께를 보이는 모델이 더 자연스러울 수 있다.
아래 데이터셋에서는 이런 기준을 함께 고려할 때 `k=5`인 모델이 가장 적절해 보인다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-12.png" width="700"/></div>

### k-평균의 한계

첫째, 초기 센트로이드의 선택에 따라 서로 다른 군집화 결과가 나올 수 있다.
따라서 좋은 결과를 얻기 위해 여러 초기화를 실행하거나 k-평균++와 같은 초기화 방식을 활용한다.

둘째, 군집수 $k$를 미리 지정해야 한다.

셋째, k-평균은 군집의 크기와 밀도가 서로 비슷하고, 군집이 구형에 가까울 때 잘 작동한다.
따라서 군집의 크기나 밀도가 크게 다르거나, 길게 늘어진 비구형 군집에서는 적절하지 않은 결과를 낼 수 있다.
예를 들어 아래 그림의 데이터는 군집이 길게 늘어진 형태이므로,
양쪽 그림에 보이는 k-평균 군집화 결과 모두 적절하지 않다.
특히 오른쪽 결과는 관성이 왼쪽보다 작지만 실제 군집 구조는 더 나쁘게 반영한다.
이는 관성이 작다는 사실만으로 군집화가 의미 있게 구성되었다고 판단할 수 없음을 보여준다.

데이터가 여러 가우스 분포의 혼합으로 잘 설명되는 타원형 군집으로 구성되어 있다면,
이어서 소개하는 가우스 혼합 모델(GMM)이 k-평균보다 더 적합할 수 있다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-13.png" width="800"/>
</div>

## DBSCAN

DBSCAN(density-based spatial clustering of applications with noise) 알고리즘은 
데이터 샘플들의 밀도가 높게 연속적으로 이어진 지역을 군집으로 지정한다.
알고리즘이 작동하는 방식은 다음과 같다.

- 각 샘플에 대해 $\varepsilon$-**이웃**에 자신을 포함하여 몇 개의 샘플이 있는지 확인한다.
    $\varepsilon$-**이웃**은 샘플을 중심으로 반경이 $\varepsilon$인 지역을 가리킨다.
- 어떤 샘플의 $\varepsilon$-이웃 안에 `min_samples` 개수 이상의 샘플이 존재한다면 해당 샘플을 
    **핵심 샘플**<font size='2'>core instance</font>라 부른다.
- 핵심 샘플의 $\varepsilon$-이웃에 포함된 샘플은 모두 동일한 군집에 속한다.
    $\varepsilon$-이웃에 포함된 다른 샘플 또한 핵심 샘플인 경우 해당 샘플의 $\varepsilon$-이웃에
    포함된 샘플도 모두 동일한 군집에 속한다.
    즉, 군집이 핵심 샘플들의 $\varepsilon$-이웃으로 이뤄진다.
- 핵심 샘플의 $\varepsilon$-이웃에 포함되지만 자신은 핵심 샘플이 아닌 샘플은
    해당 군집의 경계 샘플로 간주된다.
- 어떤 핵심 샘플로부터도 $\varepsilon$-이웃의 연쇄를 통해 도달할 수 없는 샘플은
    주변에 일부 샘플이 있더라도 이상치(잡음)로 간주된다.
    
DBSCAN 알고리즘은 군집 각각이 밀도가 낮은 지역으로 구분될 때 잘 작동한다.

### 사이킷런의 DBSCAN 모델

앞서 설명한 $\varepsilon$-이웃의 반경 `eps`와 
핵심 샘플의 `eps` 반경안에 포함되어야 할
샘플의 최소 개수를 가리키는 `min_samples` 두 개의 하이퍼파라미터를 사용한다.

**예제: 초승달 데이터 군집화**

```python
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.05)
dbscan = DBSCAN(eps=0.05, min_samples=5)
dbscan.fit(X)
```

**군집 레이블**

DBSCAN 모델이 찾은 군집 정보는 0, 1, 2, ... 등 정수 인덱스로 표기되며
각 샘플의 군집 정보는 `labels_` 속성에 저장된다.
단, -1은 이상치로 간주되는 샘플을 가리킨다.

```python
>>> dbscan.labels_[:10]
array([ 0,  2, -1, -1,  1,  0,  0,  0,  2,  5])
```

핵심 샘플들의 행 인덱스는 `core_sample_indices_` 속성에 저장된다.
예를 들어, 첫 10개의 핵심 샘플의 행 인덱스는 다음과 같다.

```python
>>> dbscan.core_sample_indices_[:10]
array([ 0,  4,  5,  6,  7,  8, 10, 11, 12, 13])
```

핵심 샘플로 구성된 데이터셋은 `components_` 속성에 저장된다.

```python
>>> dbscan.components_
array([[-0.02137124,  0.40618608],
       [-0.84192557,  0.53058695],
       [ 0.58930337, -0.32137599],
       ...,
       [ 1.66258462, -0.3079193 ],
       [-0.94355873,  0.3278936 ],
       [ 0.79419406,  0.60777171]])
```

**군집화 결과**

아래 그림은 $\varepsilon$-이웃의 반경을 0.05로 할 때(왼쪽)와 0.2로 할 때(오른쪽)의 차이를 보여준다.

| 특징| 왼쪽 그림(이웃 반경 0.05) | 오른쪽 그림(이웃 반경 0.2) |
| :---: | :---: | :---: |
| 군집수 | 2개 초과 | 2개 |
| 이상치(빨강 &#128473;) | 많음 | 없음 |

<br>

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-16.png" width="800"/></div>
<br>

**DBSCAN과 예측**

DBSCAN 모델은 `predict()` 메서드를 지원하지 않는다. 즉, 새로운 샘플에 대한 군집 예측을 지원하지 않는다.
반면에 `fit_predict()` 메서드는 지원하여 훈련셋에 대한 군집 인덱스를 예측한다.
`predict()` 메서드를 지원하지 않는 이유는 훈련 후 새 샘플을 어느 군집에 배정할지에 대한
규칙이 DBSCAN 알고리즘 자체에 정해져 있지 않기 때문이다.
필요하다면 용도에 맞는 별도의 분류 전략을 선택할 수 있다.
예를 들어 아래 코드는 `KNeighborsClassifier` 분류 모델을 핵심 샘플들을 이용하여
지도학습을 진행한다.

아래 코드에서 `dbscan`은 `eps=0.2`로 훈련된 모델을 가리킨다.
즉, 위 오른쪽 그림에서 사용된 DBSCAN 모델이다.
훈련을 위해 핵심 샘플만 이용해서 훈련하지만 당연히 모든 샘플을 이용할 수도 있다.

```python
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=50)
knn.fit(dbscan.components_, dbscan.labels_[dbscan.core_sample_indices_])
```

`knn`은 분류 모델이기에 당연히 새로운 데이터 샘플에 대해 클래스를 예측하거나
클래스별 확률을 예측할 수 있다.

```python
>>> X_new = np.array([[-0.5, 0], [0, 0.5], [1, -0.1], [2, 1]])
>>> knn.predict(X_new)
array([1, 0, 1, 0])
>>> knn.predict_proba(X_new)
array([[0.18, 0.82],
       [1. , 0. ],
       [0.12, 0.88],
       [1. , 0. ]])
```

**결정 경계**

아래 그림은 `knn` 모델을 이용하여 두 개의 그룹을 분류하는 결정 경계를 보여준다.
파랑 덧셈 기호 <font color='blue' size='4'>&#43;</font>는 이전 코드에서 지정한 `X_new` 어레이에 포함된 데이터 샘플 4 개를 가리킨다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-17a.png" width="450"/></div>

그런데 빨강 동그라미로 감싸진 두 샘플은 사실 이상치로 취급되어야 한다.
이유는 다른 데이터들로부터 너무 멀리 떨어져 있기 때문이다.

**DBSCAN의 장단점**

장점은 다음과 같다.

- 단 2개의 하이퍼파라미터만을 사용하는 단순하지만 매우 강력한 알고리즘이다. 
- 군집수를 미리 지정하지 않아도 되며, 밀도가 비슷한 임의 모양의 군집을 잘 찾을 수 있다.
- 군집과 떨어진 샘플을 이상치(잡음)로 식별할 수 있다.

반면에 단점은 다음과 같다.

- 군집들의 밀도가 서로 크게 다르거나 두 군집 사이의 영역의 밀도가 충분히 낮지 않으면
    서로 다른 군집을 제대로 분리하지 못할 수도 있다.
- 사이킷런의 구현은 `eps`가 크고 `min_samples`가 작을 때 최악의 경우
    $O(m^2)$의 메모리를 사용할 수 있어 대용량 훈련셋에서는 주의해야 한다.

## 가우스 혼합 모델

가우스 혼합 모델<font size='2'>Gaussian mixture model</font> (GMM)은 
데이터셋이 여러 개의 혼합된 가우스 분포를 따르는 샘플들로 구성되었다고 가정한다.

**가우스 분포**

데이터셋에 포함된 데이터들의 확률분포가 아래 그림에 있는 곡선들처럼 종 모양의 (정규분포) 확률밀도함수를
가질 경우 가우스 분포를 따른다고 말한다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-18.png" width="400"/></div>

예를 들어 아래 그림에 있는 데이터셋은 서로 다른 세 개의 가우스 분포를 따르는 세 개의 데이터 군집들의 혼합으로 구성된다.
세 개의 가우스 분포의 평균값, 표준편차, 샘플의 개수가 모두 다르며, 그에 따라
각각의 군집을 나타내는 타원의 모양, 위치, 타원 내의 데이터 밀도 등이 모두 다르다. 

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-13.png" width="600"/></div>

임의의 데이터셋이 어떤 가우스 분포들의 혼합으로 이뤄졌는지를 알아내서 그에 따른 
군집을 형성해야 한다.
하지만 일반적으로는 가우스 분포를 따른다는 보장도 없기에 다양한 군집 알고리즘을 활용해서
가장 좋은 알고리즘을 선택해야 한다.

여기서는 데이터셋이 여러 개의 가우스 분포로 혼합된 분포를 따른다고 가정하면서
가우스 혼합 모델의 작동 방식을 살펴 본다.

### GMM 활용

아래 코드는 위 그림에서 사용된 데이터셋에 대해 
가우스 혼합 모델(GMM)을 훈련 시켜서 데이터셋에 포함된 군집 각각을 특징 짓는 가우스 분포를 찾는다.

* `GaussianMixture`: 가우스 혼합 모델
* `n_components`: k-평균 모델의 경우처럼 군집수는 미리 지정해야 함.
* `n_init`: k-평균 모델의 경우처럼 군집의 파라미터(평균값, 공분산 등)를 무작위로 선택한 후 
    좋은 군집으로 수렴할 때까지 학습시킴. 아래 모델은 10번 시도.


```python
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=3, n_init=10, random_state=42)
gm.fit(X)
```

학습된 모델을 이용하여 아래 그림을 그릴 수 있다.

- 군집 평균: &#128473; 표시
- 결정 경계: 빨강 파선
- 밀도 등고선: 진한 파란색에 가까울수록 데이터 밀도 높음

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-19.png" width="600"/></div>

위 모델이 찾은 세 군집의 분포는 다음과 같다.

- 군집별 가중치: 군집별 데이터 수의 상대적 비율 가리킴

    ```python
    >>> gm.weights_
    array([0.39025715, 0.40007391, 0.20966893])
    ```

- 군집별 평균값

    ```python
    >>> gm.means_
    array([[ 0.05131611, 0.07521837],
           [-1.40763156, 1.42708225],
           [ 3.39893794, 1.05928897]])
    ```

- 군집별 공분산

    ```python
    array([[[ 0.68799922, 0.79606357],
            [ 0.79606357, 1.21236106]],
           [[ 0.63479409, 0.72970799],
            [ 0.72970799, 1.1610351 ]],
           [[ 1.14833585, -0.03256179],
            [-0.03256179, 0.95490931]]])
    ```

**하드/소프트 군집화**

`predict()` 메서드는 샘플이 속하는 군집을,
`predict_proba()` 메서드는 샘플이 각 군집에 속할 상대적 확률을 계산한다.

```python
>>> gm.predict(X)
array([0, 0, 1, ..., 2, 2, 2])
>>> gm.predict_proba(X).round(3)
array([[0.977, 0. , 0.023],
       [0.983, 0.001, 0.016],
       [0. , 1. , 0. ],
       ...,
       [0. , 0. , 1. ],
       [0. , 0. , 1. ],
       [0. , 0. , 1. ]])
```

**데이터 확률 밀도 측정**

`score_samples()` 메서드는 임의의 위치에서의 확률 밀도의 로그값을 측정한다. 
값이 클수록 높은 밀도를 나타낸다.

```python
>>> gm.score_samples(X).round(2)
array([-2.61, -3.57, -3.33, ..., -3.51, -4.4 , -3.81])
```

**GMM 모델 공분산 규제**

데이터셋의 차원이 크거나, 군집수가 많거나, 샘플이 적은 경우 최적의 군집화가 어려울 수 있다.
이런 경우엔 공분산 유형을 지정해서 학습을 도와줄 수 있다.
이를 위해 `covariance_type` 하이퍼파라미터를 `full` 대신 다른 값을 지정한다.
사용할 수 있는 값은 다음과 같다.
* full: 공분산에 아무런 제한 없음. 기본값으로 지정됨.
* spherical: 군집이 원형이라 가정. 지름(분산)은 다를 수 있음.
* diag: 어떤 타원형도 가능. 단, 타원의 축이 좌표축과 평행하다고 가정.
* tied: 모든 군집이 동일 모양, 동일 크기, 동일 방향을 갖는 타원형이라고 가정.

```python
gm_full = GaussianMixture(n_components=3, n_init=10,
                          covariance_type="full", random_state=42)
gm_tied = GaussianMixture(n_components=3, n_init=10,
                          covariance_type="tied", random_state=42)
gm_spherical = GaussianMixture(n_components=3, n_init=10,
                               covariance_type="spherical", random_state=42)
gm_diag = GaussianMixture(n_components=3, n_init=10,
                          covariance_type="diag", random_state=42)
```

아래 왼쪽 그림은 `"tied"`를, 오른쪽 그림은 `"spherical"`을 `covariance_type`으로 지정한 결과를 보여준다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-20.png" width="650"/></div>

아래 왼쪽 그림은 `"full"`를, 오른쪽 그림은 `"diag"`을 `covariance_type`으로 지정한 결과를 보여준다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-20a.png" width="650"/></div>

### 이상치 탐지

데이터 확률 밀도가 지정된 임곗값보다 낮은 지역에 있는 샘플을 이상치로 간주할 수 있다.
임곗값은 상황에 따라 다르게 지정된다.

예를 들어, 제품의 결함 비율이 약 2%라고 알려진 경우 훈련 샘플 가운데
추정 확률 밀도가 가장 낮은 하위 2%를 이상치 후보로 정할 수 있다.
이는 확률 밀도 값 자체가 2% 이하라는 의미가 아니라, 밀도 순위의 하위 2%를 뜻한다.
기대했던 것보다 탐지되는 결함 비율이 낮거나 높다면 백분위 기준을 조정해야 한다.

아래 그림은 추정 확률 밀도가 하위 2%에 속하는 샘플을 별표(&#x2605;)를 이용하여 이상치로 표시하였다.

```python
densities = gm.score_samples(X)
density_threshold = np.percentile(densities, 2)
anomalies = X[densities < density_threshold]
```

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-21.png" width="500"/></div>

### 군집수 지정

적절한 군집수를 미리 지정해야 하는데 일반적으로 미리 알 수 없다.
GMM의 예측 군집 레이블로 실루엣 점수를 계산할 수는 있지만,
확률 모델로서의 적합도와 복잡도를 함께 평가하려면 **이론적 정보 기준**<font size='2'>theoretical information criterion</font>을 
최소화하는 모델을 생성하는 군집수를 선택한다.

이론적 정보 기준으로 보통 다음 두 기준 중 하나를 사용하며,
값이 작을 수록 좋은 모델로 간주된다.

- BIC(Bayesian information criterion): 

    $$ \log(m)\, p - 2 \log (\hat L)$$
    
- AIC(Akaike information criterion):

    $$ 2\, p - 2 \log (\hat L)$$

위 식에 사용된 기호의 의미는 다음과 같다.

* $m$: 데이터셋 크기
* $p$: 모델이 학습해야 할 파라미터 수, 즉 모델의 복잡도를 가리킴.
* $\hat L$: 모델의 가능도 함수의 최댓값

두 식 모두 의미하는 바는 동일하다. 
첫째, 각 군집의 평균값, 표준편차 등 모델이 학습해야 할 파라미터가 많을 수록 벌칙이 가해진다.
둘째, 데이터셋에 잘 들어맞는 군집화 모델이 학습될수록 잘 학습하는 모델일수록 기준값을 낮춘다.


**군집수와 이론적 정보 기준**

아래 그림은 위 데이터셋에 대해 군집수 $k$와 AIC, BIC의 관계를 보여주며, $k=3$이 최적임을 확인해준다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-22.png" width="600"/></div>

### 베이즈 가우스 혼합 모델

`BayesianGaussianMixture` 모델은 필요한 군집수의 상한을 지정해 두고,
데이터와 사전분포 설정에 따라 실제로 사용하는 유효 군집수를 줄일 수 있다.
`n_components` 하이퍼파라미터에는 가능한 최대 군집수를 충분히 크게 지정하고,
`weight_concentration_prior`와 사전분포 유형에 따라 불필요한 군집의 가중치가
0에 매우 가까워져 해당 군집이 사실상 무시될 수 있다.

예를 들어 아래 코드는 최대 군집수를 10으로 지정하고 베이즈 가우스 혼합 모델을 훈련시킨다.

```python
from sklearn.mixture import BayesianGaussianMixture

bgm = BayesianGaussianMixture(n_components=10, n_init=10, random_state=42)
bgm.fit(X)
```

`weights_` 속성에 저장된 군집별 가중치를 확인하면 앞서 확인한 군집별 가중치 결과와 동일하게
세 개의 군집에 대해 4:2:4에 가까운 비율로 가중치를 주며 나머지 7개의 군집에 대한 가중치는
반올림하면 0으로 표시될 정도로 매우 작다.

```python
>>> bgm.weights_.round(2)
array([0.4 , 0.21, 0.4 , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])
```

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-19a.png" width="500"/></div>

**(베이즈) 가우스 혼합 모델의 장단점**

타원형 군집으로 이뤄진 데이터셋에 대해 매우 잘 작동한다.
하지만 다른 모양을 가진 데이터셋에서는 하나의 자연스러운 군집을 여러 가우스 성분으로
나누어 근사할 수 있어, 군집 해석이 의도와 달라질 수 있다.
예를 들어 초승달 데이터셋(그림 왼쪽)에 베이즈 가우스 혼합 모델을 적용하면 
아래 오른쪽 그림에서처럼 억지로 타원형 군집으로 맞추기 위해 필요 이상의 군집을 사용한다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch09/homl09-23.png" width="800"/></div>

## 연습문제

참고: [(실습) 비지도 학습](https://colab.research.google.com/github/codingalzi/handson-ml3/blob/master/practices/practice_unsupervised_learning.ipynb)